In [24]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
import warnings
warnings.filterwarnings("ignore")


In [25]:
# Environment Load
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No API Key found in .env please add!")
else:
    # Config
    genai.configure(api_key=api_key)
    
    # To get Models List
    # for m in genai.list_models():
    #     if 'generateContent' in m.supported_generation_methods:
    #         print(f"- {m.name}")
    
    # Model Initialize
    model = genai.GenerativeModel('gemini-2.5-flash')
    
    try:
        response = model.generate_content("Hello, are you working ?")
        print(f"{response.text}")
    except Exception as e:
        print(f"Error {e}")


Yes, I am! I'm always on and ready to assist.

How can I help you today?


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "../artifacts/data/policy.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f"Total Pages: {len(docs)}")

# Splitting text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
final_documents = text_splitter.split_documents(docs)

print(f"Total Chunks Created: {len(final_documents)}")
print(f"Sample Chunk: {final_documents[0].page_content[:200]}...")


Total Pages: 23
Total Chunks Created: 76
Sample Chunk: ICICI Lombard General Insurance Company Ltd. 
 
Page 1 of 23 
UIN: IRDAN115RP0001V01201920 
 
  
Annexure IV 
Stand-Alone Own Damage Private Car insurance Policy 
                                     ...


In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=api_key
        )
# Creating Vector Store
vector_store = FAISS.from_documents(final_documents, embeddings)

vector_store.save_local("../artifacts/faiss_index")
print("Vector DB Saved Successfully!")


Vector DB Saved Successfully!


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


# Load Vector DB
new_db = FAISS.load_local(
    "../artifacts/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = new_db.as_retriever()


# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=api_key
)


# Prompt
prompt = ChatPromptTemplate.from_template("""
You are an expert Insurance Assistant for 'InsureTech 360'.
Use the following pieces of context to answer the user's question.
If the answer is not in the context, just say "I don't know based on the policy document."
Don't try to make up an answer.

Context: {context}
Question: {question}

Answer:
"""
)

# RAG Runnable Chain (1.x way)
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

# Test
try:
    query = "Does this policy cover flood damage?"

    print(f"Question: {query}")

    result = rag_chain.invoke(query)

    print("🤖 AI Answer:")
    print(result.content)
except Exception as e:
    print(f"Error: {e}")


Question: Does this policy cover flood damage?
🤖 AI Answer:
Yes, this policy covers flood damage. Section I of the policy explicitly states that the company will indemnify the insured against loss or damage to the vehicle caused "By flood typhoon hurricane storm tempest inundation cyclone hailstorm frost". Other sections also list "flood inundation" as a covered peril.


In [19]:
import langchain
print(langchain.__version__)


1.2.8
